## Notebook to build an enriched case log from an event log (my version)

Motivated and parts copied from a fresh approach to analyze process outcomes ... todo: ref

In [19]:
import pandas as pd
import pm4py

## Data loading

In [ ]:
#INPUT_FILE_NAME = 'berlin-preprocessed-gesetzgebung-2006-2020_with_context.xes'
#OUTPUT_FILE_NAME = 'berlin-preprocessed-gesetzgebung-2006-2020_with_context.csv'

INPUT_FILE_NAME = "../AdvancedAnalysis/data-xes/Berlin-with-topics-and-agreement-scores-3EP-lawDraft-passedBills.xes"
OUTPUT_FILE_NAME = "Berlin-with-topics-and-agreement-scores-3EP-lawDraft-passedBills.csv"

#INPUT_FILE_NAME = "../AdvancedAnalysis/data-xes/Brandenburg-with-topics-and-agreement-scores-preprocessed-3EP-lawDraft-passedBills.xes"
#OUTPUT_FILE_NAME = 'Brandenburg-with-topics-and-agreement-scores-preprocessed-3EP-lawDraft-passedBills.csv'

#INPUT_FILE_NAME = '../AdvancedAnalysis/data-xes/Baden-Württemberg-with-topics-and-agreement-scores-preprocessed-3EP-lawDraft-passedBills.xes'
#OUTPUT_FILE_NAME = 'Baden-Württemberg-with-topics-and-agreement-scores-preprocessed-3EP-lawDraft-passedBills.csv'

CASE_ID_COL = 'case:concept:name'
TIMESTAMP_COL = 'time:timestamp'
ACTIVITY_COL = 'concept:name'

df = pm4py.read_xes(INPUT_FILE_NAME)
df = df.sort_values(TIMESTAMP_COL, ignore_index=True)
print(len(df), 'rows read from', INPUT_FILE_NAME)
#df.info()
#df.head()

parsing log, completed traces ::   0%|          | 0/501 [00:00<?, ?it/s]

4192 rows read from ../journal-extension/data-xes/Berlin-with-topics-and-agreement-scores-3EP-lawDraft-passedBills.xes


### Remove columns we do not care about

In [21]:
# remove columns that are not of interest at all
columns_to_drop = ['DBID', 'ReihNr', 'DokArt', 'DokArtL', 'lifecycle:transition', 
                   'Titel', 'DokNr', 'Sb', 'LokURL', 'HNr', 'Jg', 'NrInTyp', 
                   'DHerk', 'DHerkL', 'DokTyp', '@@index', 'DokDat', 'VkDat',
                   'case:variant', 'case:variant-index', 'case:creator',
                    'case:VSys',
                    'case:VTyp', 'case:VTypL',
                   'case:DHerk', 'case:DHerkL', 'case:Wp', 'case:NrInTyp',
                   'case:@@case_index']
df = df.drop(columns=columns_to_drop, errors='ignore')

# Case attributes

In [22]:
# Configure the number of attributes to be included
INCLUDE_DATA = False
INCLUDE_TIME = True
INCLUDE_DELAYS = True
INCLUDE_DATA_NOT_CONNECTED_TO_ACTIVITIES = True

In [23]:
case_index = {}
activity_list = []
log_starting_time = df.at[0, TIMESTAMP_COL] # this will be used to calculate the relative time
columns = df.columns.to_list()

for col in [CASE_ID_COL, ACTIVITY_COL, TIMESTAMP_COL]:
    columns.remove(col)
    
for index, row in df.iterrows():
    case_id = row[CASE_ID_COL]
    activity_name = row[ACTIVITY_COL]
    
    if not activity_name in activity_list:
        activity_list.append(activity_name)
        
    qualifier = activity_name + "."
    
    if not case_id in case_index:
        case_record = {'case_id': case_id, qualifier+'count': 1, 'event_count': 1}
        
        # add time related features
        if INCLUDE_TIME:
            case_record['start_time'] = row[TIMESTAMP_COL]
            case_record['start_time_rel'] = int((row[TIMESTAMP_COL]-log_starting_time).components.days)
            case_record['duration'] = 0
            case_record[qualifier + 'start'] = 0
        
        
        # add data related features
        for col in columns:
            if not pd.isna(row[col]):
                if INCLUDE_DATA:
                    case_record[qualifier + col] = row[col]
        
        for col in columns:
            if not pd.isna(row[col]):
                if INCLUDE_DATA_NOT_CONNECTED_TO_ACTIVITIES:
                    if (col.startswith('case:')):
                        print(col)
                        case_record[col] = row[col]
                    if (col.endswith("agreement_score")):
                        print(col)
                        case_record[col] = row[col]
        
        # store the information so that it can be updated further 
        # when more events of the same case occur (else part)
        case_index[case_id] = case_record
    
    else:
        case_record = case_index[case_id]
        case_record['event_count'] += 1
        
        # if this activity type has already been seen for this case
        # note that it is the n-th occurrence of this activity type
        # and adjust the qualifier accordingly
        # else set the count to 1
        if qualifier + "start" in case_record:
            activity_count = case_record[qualifier + 'count'] + 1
            case_record[qualifier + 'count'] = activity_count
            qualifier = activity_name + ":" + str(activity_count) + "."
        else:
            case_record[qualifier + 'count'] = 1
        
        if INCLUDE_TIME:
            duration = row[TIMESTAMP_COL] - case_record['start_time']
            case_record[qualifier + 'start'] = int(duration.components.days)
            case_record['duration'] = int(duration.components.days)
        
        
        for col in columns:
            if not pd.isna(row[col]):
                if INCLUDE_DATA:
                    case_record[qualifier + col] = row[col]
        

out_df = pd.DataFrame([case_index[case_id] for case_id in case_index])

# fill nan counts with 0
for col in out_df.columns:
    if col.endswith('.count'):
        out_df[col] = out_df[col].fillna(0)
        
if INCLUDE_DELAYS: 
    first_activities = activity_list.copy()
    while len(first_activities) > 0:
        first_act = first_activities.pop()
        second_activities = first_activities.copy()
        while len(second_activities) > 0:
            second_act = second_activities.pop()
            column_name = first_act + ':' + second_act + '.delay'
            out_df[column_name] = out_df[second_act + '.start'] - out_df[first_act + '.start']

out_df.info(verbose=True)


opposition_agreement_score
all_agreement_score
case:DokTypLFirstDoc
case:VSysL
case:VorgangsDeskriptoren
case:embedding
case:top_topics
case:top_topic
case:top_topic_similarity_score
case:government_agreement_score
case:government_agreement_score
opposition_agreement_score
all_agreement_score
case:DokTypLFirstDoc
case:VSysL
case:VorgangsDeskriptoren
case:embedding
case:top_topics
case:top_topic
case:top_topic_similarity_score
case:government_agreement_score
case:government_agreement_score
opposition_agreement_score
all_agreement_score
case:DokTypLFirstDoc
case:VSysL
case:VorgangsDeskriptoren
case:embedding
case:top_topics
case:top_topic
case:top_topic_similarity_score
case:government_agreement_score
case:government_agreement_score
opposition_agreement_score
all_agreement_score
case:DokTypLFirstDoc
case:VSysL
case:VorgangsDeskriptoren
case:embedding
case:top_topics
case:top_topic
case:top_topic_similarity_score
case:government_agreement_score
case:government_agreement_score
opposition_a

## Write output

In [24]:
output_file_name = 'cases_' + OUTPUT_FILE_NAME
out_df.to_csv(output_file_name, index=False)
print(len(out_df), 'cases written to', output_file_name)

501 cases written to cases_Berlin-with-topics-and-agreement-scores-3EP-lawDraft-passedBills.csv
